In [0]:
%pip install sentence-transformers faiss-cpu langchain langchain-community

In [0]:
dbutils.library.restartPython()

In [0]:
chunks_df = spark.sql("""
SELECT
    chunk_id,
    source,
    text
FROM pdf_rag_chunks
ORDER BY chunk_id
""")

display(chunks_df)

In [0]:
chunk_rows = chunks_df.collect()

texts = [row["text"] for row in chunk_rows]

metadata = [
    {
        "chunk_id": row["chunk_id"],
        "source": row["source"]
    }
    for row in chunk_rows
]

print("Number of text chunks:", len(texts))
print("First metadata record:", metadata[0])
print()
print("First text chunk preview:")
print(texts[0][:500])

In [0]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

In [0]:
test_embedding = embedding_model.embed_query(texts[0])

print("Embedding created")
print("Vector length:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

In [0]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_texts(
    texts=texts,
    embedding=embedding_model,
    metadatas=metadata
)

print("FAISS vector store created successfully")

In [0]:
question = "What is Tableau used for?"

In [0]:
results = vector_store.similarity_search(
    query=question,
    k=3
)

print("Question:", question)
print()
print("Top matching chunks:")
print("=" * 80)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Metadata:", doc.metadata)
    print("Text preview:")
    print(doc.page_content[:1000])
    print("-" * 80)

In [0]:
question = "What are dimensions and measures in Tableau?"

results = vector_store.similarity_search(
    query=question,
    k=3
)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:1000])
    print("-" * 80)

In [0]:
faiss_index_path = "/Volumes/workspace/365pdf/365pdfupload/faiss_intro_tableau_index"

vector_store.save_local(faiss_index_path)

print("FAISS index saved to:")
print(faiss_index_path)

In [0]:
loaded_vector_store = FAISS.load_local(
    faiss_index_path,
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS index loaded successfully")

In [0]:
question = "What is Tableau?"

results = loaded_vector_store.similarity_search(question, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:800])
    print("-" * 80)